## Data Collection - Web Scraping Best Paper Awards

This notebook **scrapes academic best paper award data** from Jeff Huang's best paper awards website (`https://jeffhuang.com/best_paper_awards/`).

### Process:

1. **Setup Selenium WebDriver** - Configures a headless Chrome browser with specific options for stability

2. **Load & Expand Content** - Navigates to the target URL and attempts to click expandable year buttons to reveal all award data

3. **Scroll to Load All Data** - Scrolls through the page 10 times to ensure dynamic content fully loads

4. **Parse HTML** - Uses BeautifulSoup to extract the HTML source code

5. **Extract Structured Data** - Iterates through all HTML tables and extracts:
   - **Year** (from colspan='3' headers)
   - **Conference** (from th with scope='row')
   - **Paper title & URL** (from links in table cells)
   - **Authors** (from cells with class='authors')

6. **Save to CSV** - Creates a pandas DataFrame and exports to `huang_awards_complete_sel.csv`

7. **Display Summary Statistics**:
   - Total awards scraped
   - Year range
   - Number of unique years/conferences
   - Awards breakdown by year
   - Top 10 conferences


### Scraping "best paper awards" Data

In [4]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from webdriver_manager.chrome import ChromeDriverManager
from bs4 import BeautifulSoup
import pandas as pd
import time

# Setup Chrome
chrome_options = Options()
chrome_options.add_argument("--headless")
chrome_options.add_argument("--no-sandbox")
chrome_options.add_argument("--disable-dev-shm-usage")
chrome_options.add_argument("--window-size=1920,1080")

driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=chrome_options)

# Go to page
url = "https://jeffhuang.com/best_paper_awards/"
driver.get(url)
time.sleep(3)

# Click ALL year buttons/links to expand collapsed sections
# Look for elements that might expand years (adjust based on actual page)
try:
    # Try clicking year headers or "expand" buttons
    year_buttons = driver.find_elements(By.XPATH, "//th[@colspan='3'] | //a[contains(@href, '#20')] | //button")
    print(f"Found {len(year_buttons)} potential year expand buttons")
    
    for btn in year_buttons:
        try:
            btn.click()
            time.sleep(0.3)
        except:
            pass
except Exception as e:
    print(f"No expandable elements found: {e}")

# Scroll to bottom multiple times to ensure everything loads
for i in range(10):
    driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
    time.sleep(1)
    print(f"Scroll {i+1}/10")

# Get HTML
html = driver.page_source
driver.quit()

# Parse with BeautifulSoup
soup = BeautifulSoup(html, 'html.parser')

# Find ALL tables (there might be one table per year, or one big table)
tables = soup.find_all('table')
print(f"Found {len(tables)} tables")

awards = []
current_year = None
current_conference = None

# Process ALL tables
for table_idx, table in enumerate(tables):
    print(f"Processing table {table_idx + 1}/{len(tables)}...")
    
    for row in table.find_all('tr'):
        # 1. Check for year header
        year_header = row.find('th', attrs={'colspan': '3'})
        if year_header:
            current_year = year_header.get_text(strip=True)
            print(f"  Found year: {current_year}")
            continue
        
        # 2. Check for conference name
        conf_th = row.find('th', attrs={'scope': 'row'})
        if conf_th:
            conf_link = conf_th.find('a')
            if conf_link:
                current_conference = conf_link.get_text(strip=True)
        
        # 3. Get paper title (first <td> or <td> with <a>)
        title_td = row.find('td')
        if not title_td:
            continue
        
        title_link = title_td.find('a')
        if title_link:
            paper_title = title_link.get_text(strip=True)
            paper_url = title_link.get('href', '')
        else:
            paper_title = title_td.get_text(strip=True)
            paper_url = ''
        
        # 4. Get authors
        authors_td = row.find('td', class_='authors')
        if authors_td:
            # Get visible text only (ignore hidden "et al." details)
            authors_text = authors_td.get_text(separator=' ', strip=True)
            # Clean up extra spaces
            authors = ' '.join(authors_text.split())
        else:
            authors = ''
        
        # Add if we have all data
        if current_year and current_conference and paper_title and len(paper_title) > 5:
            awards.append({
                'year': current_year,
                'conference': current_conference,
                'paper_title': paper_title,
                'paper_url': paper_url,
                'authors': authors
            })

# Save
df = pd.DataFrame(awards)
df.to_csv("huang_awards_complete_sel.csv", index=False)

print(f"\n{'='*60}")
print(f"✅ Scraped {len(df)} awards")
print(f"Year range: {df['year'].min()} - {df['year'].max()}")
print(f"Unique years: {sorted(df['year'].unique())}")
print(f"Conferences: {df['conference'].nunique()}")
print(f"\nAwards per year:")
print(df.groupby('year').size().sort_index(ascending=False))
print(f"\nTop 10 conferences:")
print(df['conference'].value_counts().head(10))


Found 52 potential year expand buttons
Scroll 1/10
Scroll 2/10
Scroll 3/10
Scroll 4/10
Scroll 5/10
Scroll 6/10
Scroll 7/10
Scroll 8/10
Scroll 9/10
Scroll 10/10
Found 28 tables
Processing table 1/28...
  Found year: 2023
Processing table 2/28...
  Found year: 2022
Processing table 3/28...
  Found year: 2021
Processing table 4/28...
  Found year: 2020
Processing table 5/28...
  Found year: 2019
Processing table 6/28...
  Found year: 2018
Processing table 7/28...
  Found year: 2017
Processing table 8/28...
  Found year: 2016
Processing table 9/28...
  Found year: 2015
Processing table 10/28...
  Found year: 2014
Processing table 11/28...
  Found year: 2013
Processing table 12/28...
  Found year: 2012
Processing table 13/28...
  Found year: 2011
Processing table 14/28...
  Found year: 2010
Processing table 15/28...
  Found year: 2009
Processing table 16/28...
  Found year: 2008
Processing table 17/28...
  Found year: 2007
Processing table 18/28...
  Found year: 2006
Processing table 19/28.